# Hito 2 — F1 Race Strategy Advisor: Multi-Target Modeling
### IIT414W · Group 20

> **Decision Context**: Evaluating strategy trade-offs by predicting whether a driver will finish top-10, top-5, and their exact finishing position. 
> **Targets**: 
> 1. `is_top10` (Binary: 1 = positions 1–10, 0 = position 11+ or DNF) - *Primary*
> 2. `is_top5` (Binary: 1 = positions 1–5, 0 = position 6+ or DNF) - *Expansion 1*
> 3. `finish_position` (Regression: 1 to 20) - *Expansion 2*

In [16]:
# ==========================================
# 0. Dependencies & Setup
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score, mean_absolute_error, r2_score
from sklearn.calibration import calibration_curve

SEED = 42
plt.style.use('seaborn-v0_8-whitegrid')
print('Dependencies loaded successfully.')

Dependencies loaded successfully.


---
## 1. Data Loading, Feature Engineering & Temporal Split
We enforce the locked temporal split: 
* **Train**: 2019–2021
* **Calibration**: 2022
* **Test**: 2023–2024

We use the same pre-race features (`grid_position`, `constructor_tier`) to ensure no leakage from strategy choices (which will be treated as what-if scenario inputs).

In [17]:
# ==========================================
# 1. Temporal Split & Feature Engineering
# ==========================================
DATA_PATH = './f1_strategy_race_level.csv'
df = pd.read_csv(DATA_PATH)

TRAIN_SEASONS = [2019, 2020, 2021]
CALIB_SEASONS = [2022]
TEST_SEASONS  = [2023, 2024]

tier_map = {
    'front': 1, 'top': 1, '1': 1, 1: 1,
    'upper': 2, '2': 2, 2: 2,
    'midfield': 3, '3': 3, 3: 3,
    'lower': 4, '4': 4, 4: 4,
    'backmarker': 5, '5': 5, 5: 5
}

# Apply mapping and handle missing values
df['constructor_tier'] = df['constructor_tier'].map(tier_map)

# Define Features and Targets
FEATURES = ['grid_position', 'constructor_tier']
TARGETS = ['is_top10', 'is_top5', 'finish_position']

def prepare_data(data, seasons):
    # Drop rows where any required feature or target is missing
    subset = data[data['season'].isin(seasons)].dropna(subset=FEATURES + TARGETS).copy()
    
    # Ensure features are floats for the models
    X = subset[FEATURES].astype(float)
    
    # Cast targets to correct types
    y_10 = subset['is_top10'].astype(int)
    y_5 = subset['is_top5'].astype(int)
    y_pos = subset['finish_position'].astype(float)
    
    return X, y_10, y_5, y_pos, subset

# Execute splits
X_train, y_train_10, y_train_5, y_train_pos, train_df = prepare_data(df, TRAIN_SEASONS)
X_calib, y_calib_10, y_calib_5, y_calib_pos, calib_df = prepare_data(df, CALIB_SEASONS)
X_test, y_test_10, y_test_5, y_test_pos, test_df = prepare_data(df, TEST_SEASONS)

print(f"Train shapes: X={X_train.shape}")
print(f"Calib shapes: X={X_calib.shape}")
print(f"Test shapes: X={X_test.shape}")
print("\nUnique mapped tiers in training set:", X_train['constructor_tier'].unique())

Train shapes: X=(1132, 2)
Calib shapes: X=(426, 2)
Test shapes: X=(889, 2)

Unique mapped tiers in training set: [3. 1. 5.]


---
## 2. Model Training (is_top10, is_top5, finish_position)
We train Logistic Regression models for our binary targets (to easily retrieve calibrated probabilities) and a Random Forest Regressor for `finish_position` to capture non-linearities in how starting position and car pace affect final standing.

In [18]:
# ==========================================
# 2. Model Training
# ==========================================

# Model 1: is_top10 (Logistic Regression)
lr_10 = LogisticRegression(random_state=SEED, max_iter=500)
lr_10.fit(X_train, y_train_10)
p_test_10 = lr_10.predict_proba(X_test)[:, 1]

# Model 2: is_top5 (Logistic Regression)
lr_5 = LogisticRegression(random_state=SEED, max_iter=500)
lr_5.fit(X_train, y_train_5)
p_test_5 = lr_5.predict_proba(X_test)[:, 1]

# Model 3: finish_position (Random Forest Regressor)
rf_pos = RandomForestRegressor(n_estimators=100, random_state=SEED, max_depth=5)
rf_pos.fit(X_train, y_train_pos)
pred_test_pos = rf_pos.predict(X_test)

print("Models successfully trained.")

Models successfully trained.


In [ ]:
# ==========================================
# 3. Error Analysis Slicing
# ==========================================

# Attach predictions to the test dataframe for slicing
test_df['pred_10'] = p_test_10
test_df['pred_5'] = p_test_5
test_df['pred_pos'] = pred_test_pos

# Calculate error metrics row-by-row
test_df['error_pos'] = abs(test_df['finish_position'] - test_df['pred_pos'])
test_df['brier_10'] = (test_df['is_top10'] - test_df['pred_10'])**2
test_df['brier_5'] = (test_df['is_top5'] - test_df['pred_5'])**2

print("========================================")
print("        ERROR ANALYSIS SLICING          ")
print("========================================\n")

print(">>> 1. By Strategy Type (n_stops) <<<")
display(test_df.groupby('n_stops')[['error_pos', 'brier_10', 'brier_5']].mean().round(3))

print("\n>>> 2. By Circuit Type <<<")
display(test_df.groupby('circuit_type')[['error_pos', 'brier_10', 'brier_5']].mean().round(3))

print("\n>>> 3. By Constructor Tier <<<")
display(test_df.groupby('constructor_tier')[['error_pos', 'brier_10', 'brier_5']].mean().round(3))

        ERROR ANALYSIS SLICING          

>>> 1. By Strategy Type (n_stops) <<<


,error_pos,brier_10,brier_5
n_stops,,,
0,7.042,0.220,0.069
1,3.064,0.151,0.103
2,2.714,0.124,0.091
3,3.487,0.145,0.096
4,3.030,0.125,0.103
5,2.746,0.147,0.097
6,5.543,0.363,0.172



>>> 2. By Circuit Type <<<


,error_pos,brier_10,brier_5
circuit_type,,,
permanent,2.958,0.132,0.097
semi-street,3.650,0.187,0.091
street,2.953,0.136,0.100



>>> 3. By Constructor Tier <<<


,error_pos,brier_10,brier_5
constructor_tier,,,
1,3.504,0.113,0.195
3,3.176,0.162,0.123
5,2.634,0.127,0.009


In [ ]:
print(f"Overall is_top10 Brier Score: {test_df['brier_10'].mean():.3f}")
print(f"Overall is_top5 Brier Score: {test_df['brier_5'].mean():.3f}")
print(f"Overall finish_position MAE: {test_df['error_pos'].mean():.2f}")

Overall is_top10 Brier Score: 0.140
Overall is_top5 Brier Score: 0.097
Overall finish_position MAE: 3.05
